# MIG Cement Demand Forecasting
## Step 1: Data Ingestion and Cleaning

This notebook imports the supplied SQLite data, validates its structure and quality,
investigates inventory inconsistencies, creates data-quality flags, and saves cleaned
outputs without modifying the original database.

In [118]:
import sys
import sqlite3
import pandas as pd
from pathlib import Path 

In [119]:
print("Python version:", sys.version)
print("Python location:", sys.executable)
print("Pandas version:", pd.__version__)

Python version: 3.9.6 (default, Apr 30 2025, 02:07:18) 
[Clang 17.0.0 (clang-1700.0.13.5)]
Python location: /Users/mac/Documents/CEMENT /Cement-Demand-Forecasting/cement-env/bin/python
Pandas version: 2.3.3


Path and Database Connection

In [120]:
database_path = Path("../data/raw/MIG_Cement_Records.db")
connection = sqlite3.connect(database_path)

print("Database connected successfully")

Database connected successfully


## 2. Inspect the available database tables

In [121]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
    """,
    connection
)

tables

,name
0,CementTypes
1,Operations
2,Sites


## Load the tables into pandas

In [122]:
cement_types_df = pd.read_sql_query(
    "SELECT * FROM CementTypes",
    connection
)

sites_df = pd.read_sql_query(
    "SELECT * FROM Sites",
    connection
)

operations_df = pd.read_sql_query(
    "SELECT * FROM Operations",
    connection
)

In [123]:
print("CementTypes:", cement_types_df.shape)
print("Sites:", sites_df.shape)
print("Operations:", operations_df.shape)

CementTypes: (3, 1)
Sites: (30, 4)
Operations: (32880, 11)


## PREVIEW TABLES 

In [124]:
display(cement_types_df.head())
display(sites_df.head())
display(operations_df.head())

,cement_type
0,CEM_I
1,CEM_II
2,CEM_III


,site_id,region,silo_capacity,behavior
0,SITE_001,North,448,aggressive
1,SITE_002,South,288,conservative
2,SITE_003,East,314,aggressive
3,SITE_004,South,472,conservative
4,SITE_005,South,230,aggressive


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448


## INSPECT COLUMN AND DATA TYPES 

In [125]:
print("Operations columns:")
operations_df.columns.tolist()

print("\nOperations information:")
operations_df.info()

Operations columns:

Operations information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      32880 non-null  object 
 1   site_id                   32880 non-null  object 
 2   cement_type               32880 non-null  object 
 3   planned_pour_tonnes       32880 non-null  float64
 4   consumed_tonnes           32880 non-null  float64
 5   opening_inventory_tonnes  32880 non-null  float64
 6   deliveries_tonnes         32880 non-null  float64
 7   closing_inventory_tonnes  32880 non-null  float64
 8   rain_mm                   32880 non-null  float64
 9   avg_temp_c                32880 non-null  float64
 10  silo_capacity             32880 non-null  int64  
dtypes: float64(7), int64(1), object(3)
memory usage: 2.8+ MB


Validate the expected schema 

In [126]:
expected_operations_columns = {
    "date",
    "site_id",
    "cement_type",
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity"
}

actual_operations_columns = set(
    operations_df.columns
)

missing_columns = (
    expected_operations_columns
    - actual_operations_columns
)

unexpected_columns = (
    actual_operations_columns
    - expected_operations_columns
)

print("Missing columns:", sorted(missing_columns))
print("Unexpected columns:", sorted(unexpected_columns))

if missing_columns:
    raise ValueError(
        f"Operations is missing required columns: "
        f"{sorted(missing_columns)}"
    )

Missing columns: []
Unexpected columns: []


## 5. Create clean working copies

The raw DataFrames remain unchanged. All cleaning and validation are performed on copies.

In [127]:

# create copies and standardise values 
cement_types_clean = cement_types_df.copy()
sites_clean = sites_df.copy()
operations_clean = operations_df.copy()

cement_types_clean["cement_type"] = (
    cement_types_clean["cement_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

sites_clean["site_id"] = (
    sites_clean["site_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

sites_clean["region"] = (
    sites_clean["region"]
    .astype("string")
    .str.strip()
    .str.title()
)

sites_clean["behavior"] = (
    sites_clean["behavior"]
    .astype("string")
    .str.strip()
    .str.lower()
)

operations_clean["site_id"] = (
    operations_clean["site_id"]
    .astype("string")
    .str.strip()
    .str.upper()
)

operations_clean["cement_type"] = (
    operations_clean["cement_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

operations_clean["date"] = pd.to_datetime(
    operations_clean["date"],
    errors="coerce"
)

In [128]:
# Convert date conversion
print("Date data type:", operations_clean["date"].dtype)
print("Invalid dates:", operations_clean["date"].isna().sum())
print("Earliest date:", operations_clean["date"].min())
print("Latest date:", operations_clean["date"].max())

Date data type: datetime64[ns]
Invalid dates: 0
Earliest date: 2022-01-01 00:00:00
Latest date: 2024-12-31 00:00:00


## Initial Data  Quality checks 

In [129]:
initial_quality_summary = pd.Series({
    "number_of_rows": operations_clean.shape[0],
    "number_of_columns": operations_clean.shape[1],
    "missing_cells": int(
        operations_clean.isna().sum().sum()
    ),
    "exact_duplicate_rows": int(
        operations_clean.duplicated().sum()
    ),
    "unique_dates": operations_clean["date"].nunique(),
    "unique_sites": operations_clean["site_id"].nunique(),
    "unique_cement_types": (
        operations_clean["cement_type"].nunique()
    )
})

initial_quality_summary

number_of_rows          32880
number_of_columns          11
missing_cells               0
exact_duplicate_rows        0
unique_dates             1096
unique_sites               30
unique_cement_types         3
dtype: int64

In [130]:
# Check for missing values in the dataset
missing_report = pd.DataFrame({
    "missing_count": operations_clean.isna().sum(),
    "missing_percentage": (
        operations_clean.isna().mean() * 100
    ).round(2)
})

missing_report.sort_values(
    "missing_percentage",
    ascending=False
)

,missing_count,missing_percentage
date,0,0.0
site_id,0,0.0
cement_type,0,0.0
planned_pour_tonnes,0,0.0
consumed_tonnes,0,0.0
opening_inventory_tonnes,0,0.0
deliveries_tonnes,0,0.0
closing_inventory_tonnes,0,0.0
rain_mm,0,0.0
avg_temp_c,0,0.0


In [131]:
#Hidden missing values in the dataset
text_columns = [
    "site_id",
    "cement_type"
]

missing_labels = {
    "",
    "NA",
    "N/A",
    "NULL",
    "NONE",
    "UNKNOWN",
    "-"
}

hidden_missing_results = []

for column in text_columns:
    normalised_values = (
        operations_clean[column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    hidden_missing_results.append({
        "column": column,
        "real_missing_values": int(
            normalised_values.isna().sum()
        ),
        "blank_or_placeholder_values": int(
            normalised_values.isin(
                missing_labels
            ).sum()
        )
    })

hidden_missing_report = pd.DataFrame(
    hidden_missing_results
)

hidden_missing_report

,column,real_missing_values,blank_or_placeholder_values
0,site_id,0,0
1,cement_type,0,0


## 7. Primary-key and referential-integrity checks

The documented primary key is:

`date + site_id + cement_type`

In [132]:
# checking for primary key duplicates
primary_key_columns = [
    "date",
    "site_id",
    "cement_type"
]

duplicate_primary_key_mask = (
    operations_clean.duplicated(
        subset=primary_key_columns,
        keep=False
    )
)

print(
    "Rows with duplicate primary keys:",
    int(duplicate_primary_key_mask.sum())
)

Rows with duplicate primary keys: 0


In [133]:
# Validate site IDs and Cement types 
valid_sites = set(
    sites_clean["site_id"]
)

valid_cement_types = set(
    cement_types_clean["cement_type"]
)

invalid_site_mask = (
    ~operations_clean["site_id"].isin(
        valid_sites
    )
)

invalid_cement_mask = (
    ~operations_clean["cement_type"].isin(
        valid_cement_types
    )
)

print(
    "Records with invalid site IDs:",
    int(invalid_site_mask.sum())
)

print(
    "Records with invalid cement types:",
    int(invalid_cement_mask.sum())
)

Records with invalid site IDs: 0
Records with invalid cement types: 0


## 8. Add site metadata and validate silo-capacity reference values

In [ ]:
site_metadata = (
    sites_clean[
        [
            "site_id",
            "region",
            "behavior",
            "silo_capacity"
        ]
    ].rename(
        columns={
            "silo_capacity":
            "official_silo_capacity"
        }
    )
)

# Makes the cell safe to rerun.
operations_clean = operations_clean.drop(
    columns=[
        "region",
        "behavior",
        "official_silo_capacity"
    ],
    errors="ignore"
)

operations_clean = operations_clean.merge(
    site_metadata,
    on="site_id",
    how="left",
    validate="many_to_one"
)

operations_clean.head()

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior,official_silo_capacity
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,North,aggressive,448
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,North,aggressive,448
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,North,aggressive,448
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,North,aggressive,448
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,North,aggressive,448


In [135]:
operations_clean["silo_capacity_mismatch"] = (
    operations_clean["silo_capacity"]
    != operations_clean["official_silo_capacity"]
)

print(
    "Silo-capacity reference mismatches:",
    int(
        operations_clean[
            "silo_capacity_mismatch"
        ].sum()
    )
)

Silo-capacity reference mismatches: 0


## 9. Validate numerical values

In [136]:
#  negative value checks 
non_negative_columns = [
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes",
    "rain_mm",
    "silo_capacity"
]

negative_value_summary = pd.Series({
    column: int(
        (operations_clean[column] < 0).sum()
    )
    for column in non_negative_columns
}, name="negative_value_count")

negative_value_summary.to_frame()

,negative_value_count
planned_pour_tonnes,0
consumed_tonnes,0
opening_inventory_tonnes,0
deliveries_tonnes,0
closing_inventory_tonnes,0
rain_mm,0
silo_capacity,0


In [137]:
# Additional range checks 
print(
    "Zero or negative silo capacities:",
    int(
        (
            operations_clean["silo_capacity"]
            <= 0
        ).sum()
    )
)

print(
    "Minimum temperature:",
    operations_clean["avg_temp_c"].min()
)

print(
    "Maximum temperature:",
    operations_clean["avg_temp_c"].max()
)

print(
    "Minimum rainfall:",
    operations_clean["rain_mm"].min()
)

Zero or negative silo capacities: 0
Minimum temperature: -5.0
Maximum temperature: 35.0
Minimum rainfall: 0.0


## 10. Validate the inventory-balance equation

The documented equation is:

`closing inventory = opening inventory + deliveries - consumption`

In [138]:
INVENTORY_TOLERANCE = 0.02

operations_clean[
    "expected_closing_inventory"
] = (
    operations_clean[
        "opening_inventory_tonnes"
    ]
    + operations_clean[
        "deliveries_tonnes"
    ]
    - operations_clean[
        "consumed_tonnes"
    ]
)

operations_clean[
    "inventory_balance_error"
] = (
    operations_clean[
        "closing_inventory_tonnes"
    ]
    - operations_clean[
        "expected_closing_inventory"
    ]
)

In [140]:
display(
    operations_clean[
        "inventory_balance_error"
    ].describe()
)

balance_issue_mask = (
    operations_clean[
        "inventory_balance_error"
    ].abs()
    > INVENTORY_TOLERANCE
)

print(
    "Inventory balance failures:",
    int(balance_issue_mask.sum())
)

count    3.288000e+04
mean     3.041363e-07
std      1.459112e-04
min     -1.000000e-02
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e-02
Name: inventory_balance_error, dtype: float64

Inventory balance failures: 0


### Inventory-balance finding

The maximum absolute inventory-balance difference is 0.01 tonnes.

Using a tolerance of 0.02 tonnes, the inventory-flow equation is consistent across the dataset.

## 11. Check inventory against silo capacity

In [141]:
# create overflow  measurements and flag 
operations_clean[
    "opening_overflow_tonnes"
] = (
    operations_clean[
        "opening_inventory_tonnes"
    ]
    - operations_clean[
        "silo_capacity"
    ]
).clip(lower=0)

operations_clean[
    "closing_overflow_tonnes"
] = (
    operations_clean[
        "closing_inventory_tonnes"
    ]
    - operations_clean[
        "silo_capacity"
    ]
).clip(lower=0)

operations_clean[
    "capacity_violation"
] = (
    operations_clean[
        "opening_overflow_tonnes"
    ] > 0
) | (
    operations_clean[
        "closing_overflow_tonnes"
    ] > 0
)

In [142]:
#Capacity-violation counts

print(
    "Opening inventory above capacity:",
    int(
        (
            operations_clean[
                "opening_overflow_tonnes"
            ] > 0
        ).sum()
    )
)

print(
    "Closing inventory above capacity:",
    int(
        (
            operations_clean[
                "closing_overflow_tonnes"
            ] > 0
        ).sum()
    )
)

print(
    "Rows with any capacity violation:",
    int(
        operations_clean[
            "capacity_violation"
        ].sum()
    )
)

print(
    "Percentage affected:",
    round(
        operations_clean[
            "capacity_violation"
        ].mean() * 100,
        2
    ),
    "%"
)

Opening inventory above capacity: 11427
Closing inventory above capacity: 11439
Rows with any capacity violation: 11504
Percentage affected: 34.99 %


In [143]:
# Capacity summary by site
capacity_summary_by_site = (
    operations_clean
    .groupby(
        [
            "site_id",
            "region",
            "behavior"
        ],
        as_index=False
    )
    .agg(
        records=(
            "site_id",
            "size"
        ),
        opening_capacity_violations=(
            "opening_overflow_tonnes",
            lambda values: int(
                (values > 0).sum()
            )
        ),
        closing_capacity_violations=(
            "closing_overflow_tonnes",
            lambda values: int(
                (values > 0).sum()
            )
        ),
        maximum_closing_overflow_tonnes=(
            "closing_overflow_tonnes",
            "max"
        )
    )
    .sort_values(
        "closing_capacity_violations",
        ascending=False
    )
)

display(capacity_summary_by_site)

,site_id,region,behavior,records,opening_capacity_violations,closing_capacity_violations,maximum_closing_overflow_tonnes
14,SITE_015,North,conservative,1096,1090,1091,20538.87
11,SITE_012,East,conservative,1096,1085,1086,20202.64
22,SITE_023,East,conservative,1096,1084,1085,19055.45
1,SITE_002,South,conservative,1096,1082,1083,19675.37
8,SITE_009,East,conservative,1096,1082,1083,19933.40
26,SITE_027,North,conservative,1096,1081,1082,19708.30
28,SITE_029,West,conservative,1096,1076,1077,20035.10
3,SITE_004,South,conservative,1096,1074,1075,20033.34
18,SITE_019,South,conservative,1096,1074,1075,20182.91
12,SITE_013,South,chaotic,1096,557,557,483.10


## 12. Investigate capacity violations by site behaviour

In [144]:
# Net inventory flow

operations_clean[
    "net_inventory_flow"
] = (
    operations_clean[
        "deliveries_tonnes"
    ]
    - operations_clean[
        "consumed_tonnes"
    ]
)

In [145]:
# Summary by behaviour 
behavior_flow_summary = (
    operations_clean
    .groupby("behavior")
    .agg(
        records=(
            "site_id",
            "size"
        ),
        number_of_sites=(
            "site_id",
            "nunique"
        ),
        affected_records=(
            "capacity_violation",
            "sum"
        ),
        average_deliveries=(
            "deliveries_tonnes",
            "mean"
        ),
        average_consumption=(
            "consumed_tonnes",
            "mean"
        ),
        average_net_flow=(
            "net_inventory_flow",
            "mean"
        ),
        total_net_flow=(
            "net_inventory_flow",
            "sum"
        ),
        maximum_overflow=(
            "closing_overflow_tonnes",
            "max"
        )
    )
    .round(2)
)

behavior_flow_summary

,records,number_of_sites,affected_records,average_deliveries,average_consumption,average_net_flow,total_net_flow,maximum_overflow
behavior,,,,,,,,
aggressive,15344,14,0,29.98,30.01,-0.03,-502.87,0.00
chaotic,7672,7,1767,27.07,26.82,0.25,1900.04,485.19
conservative,9864,9,9737,29.96,11.52,18.43,181827.76,20538.87


### Capacity finding

Capacity violations are systematic rather than isolated.

They are concentrated among conservative and chaotic sites. The original inventory values
are retained and flagged because replacing or capping them would break the otherwise
consistent inventory sequence.

The exact intended correction cannot be verified from the supplied data alone.

## 13. Validate daily inventory continuity

In [146]:
operations_clean = (
    operations_clean
    .sort_values(
        [
            "site_id",
            "date"
        ]
    )
    .copy()
)

operations_clean[
    "previous_closing_inventory"
] = (
    operations_clean
    .groupby("site_id")[
        "closing_inventory_tonnes"
    ]
    .shift(1)
)

operations_clean[
    "opening_continuity_error"
] = (
    operations_clean[
        "opening_inventory_tonnes"
    ]
    - operations_clean[
        "previous_closing_inventory"
    ]
)

In [147]:
# Count continuity failures
continuity_issue_mask = (
    operations_clean[
        "previous_closing_inventory"
    ].notna()
    & (
        operations_clean[
            "opening_continuity_error"
        ].abs()
        > INVENTORY_TOLERANCE
    )
)

print(
    "Inventory continuity failures:",
    int(continuity_issue_mask.sum())
)

Inventory continuity failures: 0


## 14. Check date completeness for every site

In [148]:
# Missing site-date records
expected_dates = pd.date_range(
    start=operations_clean["date"].min(),
    end=operations_clean["date"].max(),
    freq="D"
)

expected_site_dates = (
    pd.MultiIndex.from_product(
        [
            sorted(
                operations_clean[
                    "site_id"
                ].unique()
            ),
            expected_dates
        ],
        names=[
            "site_id",
            "date"
        ]
    )
)

actual_site_dates = (
    pd.MultiIndex.from_frame(
        operations_clean[
            [
                "site_id",
                "date"
            ]
        ].drop_duplicates()
    )
)

missing_site_dates = (
    expected_site_dates.difference(
        actual_site_dates
    )
)

print(
    "Missing site-date records:",
    len(missing_site_dates)
)

Missing site-date records: 0


## 15. Final data-quality summary

In [149]:
final_quality_summary = pd.Series({
    "source_rows": len(operations_df),
    "current_rows": len(operations_clean),

    "invalid_dates": int(
        operations_clean["date"]
        .isna()
        .sum()
    ),

    "missing_cells_in_original_columns": int(
        operations_clean[
            list(
                expected_operations_columns
            )
        ]
        .isna()
        .sum()
        .sum()
    ),

    "duplicate_primary_keys": int(
        operations_clean.duplicated(
            subset=primary_key_columns
        ).sum()
    ),

    "invalid_site_ids": int(
        invalid_site_mask.sum()
    ),

    "invalid_cement_types": int(
        invalid_cement_mask.sum()
    ),

    "silo_capacity_reference_mismatches": int(
        operations_clean[
            "silo_capacity_mismatch"
        ].sum()
    ),

    "negative_values": int(
        negative_value_summary.sum()
    ),

    "inventory_balance_failures": int(
        balance_issue_mask.sum()
    ),

    "inventory_continuity_failures": int(
        continuity_issue_mask.sum()
    ),

    "missing_site_dates": int(
        len(missing_site_dates)
    ),

    "capacity_violations_retained": int(
        operations_clean[
            "capacity_violation"
        ].sum()
    )
})

final_quality_summary

source_rows                           32880
current_rows                          32880
invalid_dates                             0
missing_cells_in_original_columns         0
duplicate_primary_keys                    0
invalid_site_ids                          0
invalid_cement_types                      0
silo_capacity_reference_mismatches        0
negative_values                           0
inventory_balance_failures                0
inventory_continuity_failures             0
missing_site_dates                        0
capacity_violations_retained          11504
dtype: int64

## 16. Prepare the final cleaned datasets

In [150]:
final_operations_columns = [
    "date",
    "site_id",
    "region",
    "behavior",
    "cement_type",
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes",
    "rain_mm",
    "avg_temp_c",
    "silo_capacity",
    "capacity_violation",
    "opening_overflow_tonnes",
    "closing_overflow_tonnes"
]

operations_final = (
    operations_clean[
        final_operations_columns
    ]
    .sort_values(
        [
            "site_id",
            "date",
            "cement_type"
        ]
    )
    .reset_index(drop=True)
)

sites_final = (
    sites_clean
    .drop_duplicates(
        subset=["site_id"]
    )
    .sort_values("site_id")
    .reset_index(drop=True)
)

cement_types_final = (
    cement_types_clean
    .drop_duplicates()
    .sort_values("cement_type")
    .reset_index(drop=True)
)

In [151]:
#Final dataset checks
print(
    "Operations final shape:",
    operations_final.shape
)

print(
    "Sites final shape:",
    sites_final.shape
)

print(
    "CementTypes final shape:",
    cement_types_final.shape
)

print(
    "Missing values:",
    operations_final.isna().sum().sum()
)

print(
    "Duplicate primary keys:",
    operations_final.duplicated(
        subset=primary_key_columns
    ).sum()
)

print(
    "Capacity violations retained:",
    operations_final[
        "capacity_violation"
    ].sum()
)


Operations final shape: (32880, 16)
Sites final shape: (30, 4)
CementTypes final shape: (3, 1)
Missing values: 0
Duplicate primary keys: 0
Capacity violations retained: 11504


## 17. Save cleaned data and quality reports

In [157]:
# Save the cleaned Operations dataset
operations_final.to_csv(
    "../data/processed/operations_cleaned.csv",
    index=False
)

# Save only the capacity-violation records
capacity_exceptions = operations_final[
    operations_final["capacity_violation"]
].copy()

capacity_exceptions.to_csv(
    "../reports/capacity_violation_records.csv",
    index=False
)

# Save the data-quality summary
final_quality_summary.to_csv(
    "../reports/data_quality_summary.csv",
    header=["value"]
)

print("All files saved successfully")

All files saved successfully


In [154]:
from pathlib import Path

print(
    Path(
        "../data/processed/operations_cleaned.csv"
    ).exists()
)

print(
    Path(
        "../reports/capacity_violation_records.csv"
    ).exists()
)

True
True


In [156]:
connection.close()

print("Database connection closed")

Database connection closed


## Cleaning outcome

The cleaning process retained all 32,880 operational records.

No records were removed because:

- no missing values were detected;
- no duplicate primary keys were detected;
- all site IDs and cement types were valid;
- no invalid negative operational quantities were detected;
- silo-capacity values matched the Sites reference table;
- inventory arithmetic was valid within a 0.02-tonne tolerance;
- daily inventory continuity was maintained;
- no site-date records were missing.

However, 11,504 operational records had opening or closing inventory above the
documented silo capacity.

These values were not silently replaced or capped. They were retained and identified
using:

- `capacity_violation`;
- `opening_overflow_tonnes`;
- `closing_overflow_tonnes`.

A separate capacity-exception report was also generated for further operational review.